In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("credit_risk_dataset.csv")

In [3]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
df.shape

(32581, 12)

In [5]:
df.columns

Index(['person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
person_age,32581.0,27.734600,6.348078,20.00,23.00,26.00,30.00,144.00
person_income,32581.0,66074.848470,61983.119168,4000.00,38500.00,55000.00,79200.00,6000000.00
person_emp_length,31686.0,4.789686,4.142630,0.00,2.00,4.00,7.00,123.00
loan_amnt,32581.0,9589.371106,6322.086646,500.00,5000.00,8000.00,12200.00,35000.00
loan_int_rate,29465.0,11.011695,3.240459,5.42,7.90,10.99,13.47,23.22
loan_status,32581.0,0.218164,0.413006,0.00,0.00,0.00,0.00,1.00
loan_percent_income,32581.0,0.170203,0.106782,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32581.0,5.804211,4.055001,2.00,3.00,4.00,8.00,30.00


In [8]:
# ==========================================================
# Duplicate Records
# ==========================================================

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 165


In [9]:
df = df.drop_duplicates()

In [10]:
# ==========================================================
# Check Age Distribution
# ==========================================================

print(df["person_age"].describe())

print("\nMaximum Age:", df["person_age"].max())

print("\nBorrowers aged above 100:")

df[df["person_age"] > 100]

count    32416.000000
mean        27.747008
std          6.354100
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max        144.000000
Name: person_age, dtype: float64

Maximum Age: 144

Borrowers aged above 100:


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25


In [11]:
# ==========================================================
# Remove Unrealistic Age Values
# ==========================================================

print("Rows before removal:", df.shape[0])

df = df[df["person_age"] <= 100]

print("Rows after removal:", df.shape[0])

Rows before removal: 32416
Rows after removal: 32411


In [12]:
print(df["person_age"].describe())

print("Maximum Age:", df["person_age"].max())

count    32411.000000
mean        27.730369
std          6.210448
min         20.000000
25%         23.000000
50%         26.000000
75%         30.000000
max         94.000000
Name: person_age, dtype: float64
Maximum Age: 94


In [13]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              887
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3094
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [14]:
# ==========================================================
# Variable Types
# ==========================================================

numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Variables")
print(numerical_columns)

print("\nCategorical Variables")
print(categorical_columns)

Numerical Variables
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_cred_hist_length']

Categorical Variables
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [15]:
for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())


person_home_ownership
person_home_ownership
RENT        16375
MORTGAGE    13367
OWN          2563
OTHER         106
Name: count, dtype: int64

loan_intent
loan_intent
EDUCATION            6409
MEDICAL              6042
VENTURE              5680
PERSONAL             5497
DEBTCONSOLIDATION    5189
HOMEIMPROVEMENT      3594
Name: count, dtype: int64

loan_grade
loan_grade
A    10703
B    10384
C     6436
D     3620
E      963
F      241
G       64
Name: count, dtype: int64

cb_person_default_on_file
cb_person_default_on_file
N    26681
Y     5730
Name: count, dtype: int64


In [16]:
# ==========================================================
# Handle Missing Numerical Values
# ==========================================================

for col in numerical_columns:

    df[col].fillna(df[col].median(), inplace=True)

In [17]:
# ==========================================================
# Handle Missing Categorical Values
# ==========================================================

for col in categorical_columns:

    df[col].fillna(df[col].mode()[0], inplace=True)

In [18]:
df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [19]:
df_clean=df

In [20]:
# ==========================================================
# One-Hot Encoding
# ==========================================================

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

In [21]:
df["loan_status"].value_counts()

loan_status
0    25322
1     7089
Name: count, dtype: int64

In [22]:
df["loan_status"].value_counts(normalize=True) * 100

loan_status
0    78.127796
1    21.872204
Name: proportion, dtype: float64

In [23]:
# ==========================================================
# Features and Target
# ==========================================================

X = df.drop(columns=["loan_status"])

y = df["loan_status"]

In [24]:
# ==========================================================
# Train-Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [25]:
df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
0,22,59000,123.0,35000,16.02,1,0.59,3,0,0,...,0,1,0,0,0,1,0,0,0,1
1,21,9600,5.0,1000,11.14,0,0.10,2,0,1,...,0,0,0,1,0,0,0,0,0,0
2,25,9600,1.0,5500,12.87,1,0.57,3,0,0,...,1,0,0,0,1,0,0,0,0,0
3,23,65500,4.0,35000,15.23,1,0.53,2,0,0,...,1,0,0,0,1,0,0,0,0,0
4,24,54400,8.0,35000,14.27,1,0.55,4,0,0,...,1,0,0,0,1,0,0,0,0,1


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32411 entries, 0 to 32580
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   person_age                   32411 non-null  int64  
 1   person_income                32411 non-null  int64  
 2   person_emp_length            32411 non-null  float64
 3   loan_amnt                    32411 non-null  int64  
 4   loan_int_rate                32411 non-null  float64
 5   loan_status                  32411 non-null  int64  
 6   loan_percent_income          32411 non-null  float64
 7   cb_person_cred_hist_length   32411 non-null  int64  
 8   person_home_ownership_OTHER  32411 non-null  int64  
 9   person_home_ownership_OWN    32411 non-null  int64  
 10  person_home_ownership_RENT   32411 non-null  int64  
 11  loan_intent_EDUCATION        32411 non-null  int64  
 12  loan_intent_HOMEIMPROVEMENT  32411 non-null  int64  
 13  loan_intent_MEDICAL  

In [27]:
numerical_columns.remove("loan_status")

In [28]:
numerical_columns

['person_age',
 'person_income',
 'person_emp_length',
 'loan_amnt',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_cred_hist_length']

In [29]:
# ==========================================================
# Feature Scaling
# ==========================================================

scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [30]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (25928, 22)
Testing Features  : (6483, 22)
Training Target   : (25928,)
Testing Target    : (6483,)


In [31]:
X_train.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
30018,2.477919,-0.417085,3.226576,-0.249892,-0.550034,0.091489,2.275058,0,1,0,...,1,0,0,0,0,0,0,0,0,0
24725,-0.112438,0.177428,1.032289,0.067296,-0.449207,-0.376331,0.792241,0,0,1,...,0,0,0,1,0,0,0,0,0,0
13354,-0.921924,0.465096,-0.918189,-1.122159,0.718441,-1.311972,-0.937713,0,0,0,...,0,0,0,0,1,0,0,0,0,0
30534,2.154124,2.766440,-0.918189,-0.329189,-1.620107,-1.218408,2.027922,0,0,1,...,0,1,0,0,0,0,0,0,0,0
25103,-0.112438,-0.589686,0.788479,0.384484,-0.358137,1.588515,0.545105,0,0,1,...,0,1,0,1,0,0,0,0,0,0


In [32]:
y_train.head()

30018    0
24725    0
13354    0
30534    0
25103    1
Name: loan_status, dtype: int64

In [33]:
from pathlib import Path

DOWNLOADS_DIR = Path.home() / "Downloads"
X_train.to_csv(DOWNLOADS_DIR/"X_train.csv", index=False)
X_test.to_csv(DOWNLOADS_DIR/"X_test.csv", index=False)

y_train.to_csv(DOWNLOADS_DIR/"y_train.csv", index=False)
y_test.to_csv(DOWNLOADS_DIR/"y_test.csv", index=False)

In [34]:
df_clean.to_csv(DOWNLOADS_DIR/'clean_credit.csv',index=False)